# Blending saved surface-forcing files

[Blending a regional and a global surface-forcing product](blend_surface_forcing.ipynb) shows how `blend_surface_forcing` works, using two `SurfaceForcing` objects in memory. For a run of months or years, it is more practical to build each product once, save it to NetCDF, and blend the saved files: each product can be made separately (and re-used), and the blend reads the files lazily, a day at a time, so memory use does not grow with the length of the run.

This notebook does that for one day that crosses a month boundary, so each product is saved as two monthly files, as it would be for a long run.

In [ ]:
import tempfile
from datetime import datetime
from pathlib import Path

import netCDF4
import numpy as np
import xarray as xr

from roms_tools import Grid, SurfaceForcing
from roms_tools.scripts.blend_surface_forcing import (
    blend_surface_forcing,
    save_blended_surface_forcing,
)

workdir = Path(tempfile.mkdtemp(prefix="blend_example_"))

## Build and save the two products

The grid is the same one used in the other blending example: it straddles the northern edge of CONUS404. Both products are built for the same times and `model_reference_date`, on the same grid, with the same explicit `coarse_grid_mode`, and saved with `SurfaceForcing.save`, which writes one file per month.

In [ ]:
grid = Grid(
    nx=16,
    ny=16,
    size_x=700,
    size_y=700,
    center_lon=-130,
    center_lat=52,  # straddles the CONUS404 northern limit at this longitude
    rot=0,
)

common = dict(
    grid=grid,
    start_time=datetime(2020, 6, 30),
    end_time=datetime(2020, 7, 1),
    type="physics",
    coarse_grid_mode="never",  # the same for both products
    use_dask=True,
)

In [ ]:
%%time
conus404 = SurfaceForcing(**common, source={"name": "CONUS404"}, correct_radiation=False)
conus404_files = conus404.save(workdir / "conus404")

2026-11-07 09:58:05 - INFO - No path specified for CONUS404 source; defaulting to the HyTEST zarr store on the USGS/OSN pod (s3://hytest/conus404/conus404_hourly.zarr).
2026-11-07 09:58:12 - INFO - Data will be interpolated onto the fine grid. Remember to set `interp_bulk_frc = .false.` in the `&SURF_FRC_SETTINGS` group of your ROMS `namelist.nml` file.
2026-11-07 09:58:19 - INFO - Writing the following NetCDF files:
/tmp/blend_example_8mjptt5u/conus404_202006.nc
/tmp/blend_example_8mjptt5u/conus404_202007.nc



[########################################] | 100% Completed | 13.58 s

[########################################] | 100% Completed | 11.36 s
CPU times: user 21.6 s, sys: 6.01 s, total: 27.6 s
Wall time: 40.3 s


In [ ]:
%%time
era5 = SurfaceForcing(**common, source={"name": "ERA5"}, correct_radiation=True)
era5_files = era5.save(workdir / "era5")

2026-11-07 09:58:45 - INFO - No path specified for ERA5 source; defaulting to ARCO ERA5 dataset on Google Cloud.
2026-11-07 09:58:46 - INFO - gcsfs experimental features enabled via GCSFS_EXPERIMENTAL_ZB_HNS_SUPPORT.
2026-11-07 10:00:08 - INFO - Data will be interpolated onto the fine grid. Remember to set `interp_bulk_frc = .false.` in the `&SURF_FRC_SETTINGS` group of your ROMS `namelist.nml` file.
2026-11-07 10:00:24 - INFO - Writing the following NetCDF files:
/tmp/blend_example_8mjptt5u/era5_202006.nc
/tmp/blend_example_8mjptt5u/era5_202007.nc



[########################################] | 100% Completed | 9.87 s

[########################################] | 100% Completed | 4.20 s
CPU times: user 1min 49s, sys: 17.9 s, total: 2min 7s
Wall time: 2min 9s


In [ ]:
sorted(p.name for p in workdir.glob("*.nc"))

['conus404_202006.nc',
 'conus404_202007.nc',
 'era5_202006.nc',
 'era5_202007.nc']

## Blend all the files at once

Either argument of `blend_surface_forcing` can be a path, a glob pattern, or a list of paths. The files are opened lazily, so nothing is read until the result is computed or saved.

When the inputs are files, the grid is not known from them, so pass `grid`, the grid both products were built on. It is required: its land mask decides which points may be filled, so that only land points are filled and a `NaN` left at an ocean point raises an error, and its spacing converts `blend_width_km` to grid cells. Leaving it out is an error, raised before any data are read:

In [ ]:
try:
    blend_surface_forcing(workdir / "conus404_*.nc", workdir / "era5_*.nc")
except ValueError as err:
    print(err)

Pass `grid=`: the roms_tools Grid the products were built on (or a dataset with `pm`, `pn` and `mask_rho`). It is needed for the land mask, so that only land points are filled, and for the grid spacing. It can be left out only when `primary` or `fallback` is a SurfaceForcing.


In [ ]:
blended = blend_surface_forcing(
    str(workdir / "conus404_*.nc"),
    str(workdir / "era5_*.nc"),
    grid=grid,
    blend_width_km=150,
)
blended

2026-11-07 10:00:55 - INFO - Primary covers 77.7% of the target ocean points; the remainder comes from the fallback.


<xarray.Dataset> Size: 255kB
Dimensions:   (time: 28, eta_rho: 18, xi_rho: 18)
Coordinates:
  * time      (time) float64 224B 7.486e+03 7.486e+03 ... 7.487e+03 7.487e+03
    abs_time  (time) datetime64[ns] 224B dask.array<chunksize=(24,), meta=np.ndarray>
    rad_time  (time) float64 224B dask.array<chunksize=(24,), meta=np.ndarray>
Dimensions without coordinates: eta_rho, xi_rho
Data variables:
    uwnd      (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    vwnd      (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    swrad     (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    lwrad     (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    Tair      (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    rain      (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
    qair      (time, eta_rho, xi_rho) float32 36kB dask.array<chunksize=(24, 18, 18), meta=np.ndarray>
Attributes: (12/20)
    title:                       ROMS surface forcing file created by ROMS-Tools
    roms_tools_version:          5.3.1.dev7+g792e863b3.d20261107
    start_time:                  2020-06-30 00:00:00
    end_time:                    2020-07-01 00:00:00
    source:                      CONUS404
    correct_radiation:           False
    ...                          ...
    fallback_source:             ERA5
    fallback_correct_radiation:  True
    blend_method:                feathered
    blend_width_km:              150.0
    blend_profile:               smoothstep
    primary_coverage_fraction:   0.7767

## Write the result

`save_blended_surface_forcing` writes the blended dataset the way `SurfaceForcing.save` writes a single product: one file per month, with the encodings ROMS needs. Use it rather than `to_netcdf`: a blend of in-memory `SurfaceForcing` objects carries a datetime `abs_time`, which `to_netcdf` writes as a 64-bit integer, and a ROMS build that uses ParallelIO cannot read 64-bit integers. Such a build also needs the CDF5 format, `"NETCDF3_64BIT_DATA"`.

In [ ]:
%%time
blended_files = save_blended_surface_forcing(
    blended, workdir / "blended", format="NETCDF3_64BIT_DATA"
)
[p.name for p in blended_files]

2026-11-07 10:00:56 - INFO - Writing the following NetCDF files:
/tmp/blend_example_8mjptt5u/blended_202006.nc
/tmp/blend_example_8mjptt5u/blended_202007.nc



[########################################] | 100% Completed | 205.13 ms

[########################################] | 100% Completed | 103.15 ms
CPU times: user 288 ms, sys: 46.1 ms, total: 334 ms
Wall time: 421 ms


['blended_202006.nc', 'blended_202007.nc']

## Check the files before running ROMS

A few checks catch most problems: the format and variable types are what the ROMS build expects, the time axis is complete, there are no `NaN`s left (a `NaN` at an ocean point would already have stopped the save), and the blend attributes record what was done.

In [ ]:
for path in blended_files:
    with netCDF4.Dataset(path) as nc:
        int64 = [v for v in nc.variables if nc[v].dtype == np.int64]
        print(
            f"{path.name}: {nc.data_model}, {nc.dimensions['time'].size} records, "
            f"Tair {nc['Tair'].dtype}, abs_time {nc['abs_time'].dtype}, int64 variables: {int64 or 'none'}"
        )

ds = xr.open_mfdataset(blended_files, decode_timedelta=False)
print("NaN values left:", int(sum(ds[v].isnull().sum() for v in ds.data_vars)))
print("time steps (h):", np.unique(np.diff(ds["abs_time"].values).astype("timedelta64[m]")).astype(int) / 60)
{k: ds.attrs[k] for k in ("source", "fallback_source", "blend_method", "blend_width_km", "primary_coverage_fraction")}

blended_202006.nc: NETCDF3_64BIT_DATA, 25 records, Tair float32, abs_time float64, int64 variables: none
blended_202007.nc: NETCDF3_64BIT_DATA, 3 records, Tair float32, abs_time float64, int64 variables: none
NaN values left: 0
time steps (h): [1.]


{'source': 'CONUS404',
 'fallback_source': 'ERA5',
 'blend_method': 'feathered',
 'blend_width_km': '150.0',
 'primary_coverage_fraction': '0.7767'}

## Long runs: blend month by month

For a multi-year run, blend and write one month at a time rather than opening every file at once. Pair each primary file with the fallback file for the same month: the `exact` time alignment (the default) requires the fallback to have every time stamp of the primary, and files written by `SurfaceForcing.save` for the same dates have identical stamps. If a fallback product was made separately, with time stamps that do not line up exactly, pass `time_align="nearest"` (with `time_tolerance`) or `time_align="linear"`.

In [ ]:
%%time
for primary_file in sorted(workdir.glob("conus404_*.nc")):
    month = primary_file.stem.rsplit("_", 1)[1]
    fallback_file = workdir / f"era5_{month}.nc"
    monthly = blend_surface_forcing(primary_file, fallback_file, grid=grid, blend_width_km=150)
    written = save_blended_surface_forcing(
        monthly, workdir / f"blended_by_month_{month}", group=False, format="NETCDF3_64BIT_DATA"
    )
    print(month, "->", [p.name for p in written])

2026-11-07 10:00:56 - INFO - Primary covers 77.7% of the target ocean points; the remainder comes from the fallback.
2026-11-07 10:00:57 - INFO - Writing the following NetCDF files:
/tmp/blend_example_8mjptt5u/blended_by_month_202006.nc



[########################################] | 100% Completed | 204.78 ms
202006 -> ['blended_by_month_202006.nc']


2026-11-07 10:00:57 - INFO - Primary covers 77.7% of the target ocean points; the remainder comes from the fallback.
2026-11-07 10:00:58 - INFO - Writing the following NetCDF files:
/tmp/blend_example_8mjptt5u/blended_by_month_202007.nc



[########################################] | 100% Completed | 102.16 ms
202007 -> ['blended_by_month_202007.nc']
CPU times: user 1.38 s, sys: 86.5 ms, total: 1.46 s
Wall time: 1.53 s


The month-by-month files match the files written from the single blend:

In [ ]:
for path in blended_files:
    month = path.stem.rsplit("_", 1)[1]
    a = xr.open_dataset(path, decode_timedelta=False)
    b = xr.open_dataset(workdir / f"blended_by_month_{month}.nc", decode_timedelta=False)
    xr.testing.assert_identical(a.drop_attrs(), b.drop_attrs())
    print(month, "identical")

202006 identical
202007 identical


<div class="alert alert-info">

**Reproducibility.** A blended file cannot be regenerated from a single YAML file. Keep each product's YAML (`SurfaceForcing.to_yaml`) and the arguments passed to `blend_surface_forcing`; together they describe the blended file, whose attributes also record the blend method, width and fallback source.

**Month boundaries.** `SurfaceForcing.save` writes each record to the file for its month, so the monthly files of two products made for the same dates have the same records, and each blended month needs only that month's two files.

</div>